In [2]:
import pandas as pd
import numpy as np

## Missing Data in pandas

Missing data is represented differently depending on the dtype:

- **float64** columns use `NaN` (IEEE floating-point Not a Number) as the sentinel value.
- **object** columns treat both `np.nan` and Python's `None` as NA.
- When an integer column contains `None`, pandas upcasts it to `float64` and uses `NaN`.

### Checking for Missing Values

| Method | Returns |
|--------|--------|
| `isna()` | Boolean — `True` where value is NA |
| `notna()` | Boolean — `True` where value is **not** NA |

Both `np.nan` and `None` are treated as NA by `isna()`.

### NA Handling Methods Reference

| Method | Description |
|--------|-------------|
| `dropna` | Remove labels where values are missing, with configurable thresholds |
| `fillna` | Fill missing values with a constant, dict, or interpolation method |
| `isna` | Boolean mask — `True` for NA values |
| `notna` | Boolean mask — `True` for non-NA values |

## Key Points

- pandas uses `NaN` as the standard sentinel for missing numeric data.
- Both `np.nan` and `None` are treated as NA in object-dtype Series.
- `None` in a float64 Series is silently converted to `NaN`.
- All descriptive statistics exclude NA values by default.

In [3]:
float_data = pd.Series([1.2, -3.5, np.nan, 0])

float_data

0    1.2
1   -3.5
2    NaN
3    0.0
dtype: float64

In [4]:
float_data.isna()

0    False
1    False
2     True
3    False
dtype: bool

In [5]:
# Both np.nan and None are treated as NA in object dtype
string_data = pd.Series(["aardvark", np.nan, None, "avocado"])

string_data.isna()

0    False
1     True
2     True
3    False
dtype: bool

In [6]:
# None in a float64 Series is upcast to NaN
float_data = pd.Series([1, 2, None], dtype="float64")

print(float_data)
print()
print(float_data.isna())

0    1.0
1    2.0
2    NaN
dtype: float64

0    False
1    False
2     True
dtype: bool


## Filtering Out Missing Data — `dropna`

`dropna` returns a new object with missing values removed. The original is not modified.

### On a Series

Returns only the non-null values. Equivalent to `data[data.notna()]`.

### On a DataFrame

| Argument | Effect |
|----------|--------|
| `dropna()` | Drop any row containing at least one NA |
| `dropna(how="all")` | Drop only rows where **all** values are NA |
| `dropna(axis="columns", how="all")` | Drop columns where all values are NA |
| `dropna(thresh=n)` | Keep rows with at least `n` non-NA values |

`axis` and `how` can be combined freely.

## Key Points

- `dropna` returns a new object — does not modify in place.
- Default: drops any row with at least one NA.
- `how="all"` only removes rows/columns that are entirely NA.
- `thresh=n` keeps rows with at least `n` valid (non-NA) values.
- `axis="columns"` applies the same logic to columns instead of rows.

In [7]:
data = pd.Series([1, np.nan, 3.5, np.nan, 7])

data.dropna()

0    1.0
2    3.5
4    7.0
dtype: float64

In [8]:
data[data.notna()]  # equivalent to dropna()

0    1.0
2    3.5
4    7.0
dtype: float64

In [9]:
data = pd.DataFrame([
    [1.,  6.5, 3.],
    [1.,  np.nan, np.nan],
    [np.nan, np.nan, np.nan],
    [np.nan, 6.5, 3.]
])

data

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
2,NaN,NaN,NaN
3,NaN,6.5,3.0


In [10]:
data.dropna()  # drop any row with at least one NA

,0,1,2
0,1.0,6.5,3.0


In [11]:
data.dropna(how="all")  # drop only all-NA rows

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
3,NaN,6.5,3.0


In [12]:
data[4] = np.nan  # add an all-NA column

data.dropna(axis="columns", how="all")  # drop all-NA columns

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
2,NaN,NaN,NaN
3,NaN,6.5,3.0


In [13]:
df = pd.DataFrame(np.random.standard_normal((7, 3)))
df.iloc[:4, 1] = np.nan
df.iloc[:2, 2] = np.nan

df

,0,1,2
0,-1.552721,NaN,NaN
1,0.615083,NaN,NaN
2,2.042479,NaN,0.850466
3,0.401020,NaN,-1.150182
4,-0.938494,0.694339,0.815632
5,-1.427329,0.607667,1.101350
6,-1.102707,-0.659265,0.613841


In [14]:
df.dropna(thresh=2)  # keep rows with at least 2 non-NA values

,0,1,2
2,2.042479,NaN,0.850466
3,0.401020,NaN,-1.150182
4,-0.938494,0.694339,0.815632
5,-1.427329,0.607667,1.101350
6,-1.102707,-0.659265,0.613841


## Filling In Missing Data — `fillna`

Instead of dropping missing values, `fillna` replaces them in place.

### Fill Strategies

| What you pass | Effect |
|---------------|--------|
| Scalar | Replace every NA with that value |
| Dict `{col: value}` | Different fill value per column |
| `method="ffill"` | Forward-fill: propagate last valid value forward |
| `method="bfill"` | Backward-fill: propagate next valid value backward |

### `limit`

When using `ffill` or `bfill`, `limit=n` caps how many consecutive NA positions are filled.

### Data Imputation

`fillna` can also be used with computed statistics for simple imputation:

```python
data.fillna(data.mean())   # replace NA with column means
data.fillna(data.median()) # replace NA with column medians
```

### `fillna` Argument Reference

| Argument | Description |
|----------|-------------|
| `value` | Scalar or dict to fill missing values |
| `method` | `"ffill"` or `"bfill"` |
| `axis` | Axis to fill along: `"index"` (default) or `"columns"` |
| `limit` | Max consecutive NA periods to fill when using `method` |

## Key Points

- `fillna(scalar)` replaces all NA with one value.
- `fillna(dict)` allows different fill values per column.
- `fillna(method="ffill")` propagates the last valid value forward.
- `limit` caps how many consecutive NAs are filled by `ffill`/`bfill`.
- Use `fillna(data.mean())` for simple mean imputation.

In [15]:
df.fillna(0)  # fill all NA with 0

,0,1,2
0,-1.552721,0.000000,0.000000
1,0.615083,0.000000,0.000000
2,2.042479,0.000000,0.850466
3,0.401020,0.000000,-1.150182
4,-0.938494,0.694339,0.815632
5,-1.427329,0.607667,1.101350
6,-1.102707,-0.659265,0.613841


In [16]:
df.fillna({1: 0.5, 2: 0})  # per-column fill values

,0,1,2
0,-1.552721,0.500000,0.000000
1,0.615083,0.500000,0.000000
2,2.042479,0.500000,0.850466
3,0.401020,0.500000,-1.150182
4,-0.938494,0.694339,0.815632
5,-1.427329,0.607667,1.101350
6,-1.102707,-0.659265,0.613841


In [17]:
df = pd.DataFrame(np.random.standard_normal((6, 3)))
df.iloc[2:, 1] = np.nan
df.iloc[4:, 2] = np.nan

df

,0,1,2
0,0.369193,-1.840886,-0.513287
1,-1.682653,2.399807,-1.019687
2,0.727294,NaN,-1.013571
3,-0.729672,NaN,-0.576803
4,-0.415457,NaN,NaN
5,0.855809,NaN,NaN


In [18]:
df.fillna(method="ffill")  # forward-fill all NA gaps

/tmp/ipykernel_10307/4076583901.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method="ffill")  # forward-fill all NA gaps


,0,1,2
0,0.369193,-1.840886,-0.513287
1,-1.682653,2.399807,-1.019687
2,0.727294,2.399807,-1.013571
3,-0.729672,2.399807,-0.576803
4,-0.415457,2.399807,-0.576803
5,0.855809,2.399807,-0.576803


In [19]:
df.fillna(method="ffill", limit=2)  # forward-fill at most 2 consecutive NAs

/tmp/ipykernel_10307/4263426860.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method="ffill", limit=2)  # forward-fill at most 2 consecutive NAs


,0,1,2
0,0.369193,-1.840886,-0.513287
1,-1.682653,2.399807,-1.019687
2,0.727294,2.399807,-1.013571
3,-0.729672,2.399807,-0.576803
4,-0.415457,NaN,-0.576803
5,0.855809,NaN,-0.576803


In [20]:
# Mean imputation
data = pd.Series([1., np.nan, 3.5, np.nan, 7])

data.fillna(data.mean())

0    1.000000
1    3.833333
2    3.500000
3    3.833333
4    7.000000
dtype: float64